In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "8"

In [2]:
from tqdm.auto import tqdm
from typing import List, Tuple
import re
DEVICE = 'cuda:0'
import torch

In [3]:
import sys
sys.path.append('../../..')
print(os.path.realpath("."))

/mount/arbeitsdaten41/projekte/asr-2/vaethdk/cts_generated_v3/generation/reimburse/llama


In [4]:
a = torch.zeros(10,10,device=DEVICE)

In [3]:
# !GITHUB_ACTIONS=true pip install auto-gptq

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from auto_gptq import AutoGPTQForCausalLM, BaseQuantizeConfig
from data.dataset import ReimburseGraphDataset, DataAugmentationLevel, NodeType, DialogNode, Question

/mount/arbeitsdaten/asr-2/vaethdk/virtualenvs/cts_al/lib64/python3.10/site-packages/auto_gptq/nn_modules/triton_utils/kernels.py:411: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  def forward(ctx, input, qweight, scales, qzeros, g_idx, bits, maxq):
/mount/arbeitsdaten/asr-2/vaethdk/virtualenvs/cts_al/lib64/python3.10/site-packages/auto_gptq/nn_modules/triton_utils/kernels.py:419: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
/mount/arbeitsdaten/asr-2/vaethdk/virtualenvs/cts_al/lib64/python3.10/site-packages/auto_gptq/nn_modules/triton_utils/kernels.py:461: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float16)
CUDA extension not installed.
CUDA 

In [6]:
model_name_or_path = "TheBloke/upstage-llama-30b-instruct-2048-GPTQ"
model_basename = "gptq_model-4bit--1g"

model = AutoModelForCausalLM.from_pretrained(model_name_or_path,
                                             device_map="auto",
                                             trust_remote_code=False,
                                             revision="main",
                                             cache_dir="/mount/arbeitsdaten/asr-2/vaethdk/resources/weights/")
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path,
                                          use_fast=True,
                                          cache_dir="/mount/arbeitsdaten/asr-2/vaethdk/resources/weights/")


model.safetensors:   0%|          | 0.00/16.9G [00:00<?, ?B/s]

/mount/arbeitsdaten/asr-2/vaethdk/virtualenvs/cts_al/lib64/python3.10/site-packages/transformers/modeling_utils.py:4664: FutureWarning: `_is_quantized_training_enabled` is going to be deprecated in transformers 4.39.0. Please use `model.hf_quantizer.is_trainable` instead
  warnings.warn(
Some weights of the model checkpoint at TheBloke/upstage-llama-30b-instruct-2048-GPTQ were not used when initializing LlamaForCausalLM: ['model.layers.0.mlp.down_proj.bias', 'model.layers.0.mlp.gate_proj.bias', 'model.layers.0.mlp.up_proj.bias', 'model.layers.0.self_attn.k_proj.bias', 'model.layers.0.self_attn.o_proj.bias', 'model.layers.0.self_attn.q_proj.bias', 'model.layers.0.self_attn.v_proj.bias', 'model.layers.1.mlp.down_proj.bias', 'model.layers.1.mlp.gate_proj.bias', 'model.layers.1.mlp.up_proj.bias', 'model.layers.1.self_attn.k_proj.bias', 'model.layers.1.self_attn.o_proj.bias', 'model.layers.1.self_attn.q_proj.bias', 'model.layers.1.self_attn.v_proj.bias', 'model.layers.10.mlp.down_proj.bias'

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.


### System:
{System}

### User:
{User}

### Assistant:
{Assistant}

In [17]:
system = """You are a helpful assistant creating a list of FAQ-style questions from given facts.
Only generate questions that can be answered by the given facts, without any external knowledge.
Remove some information, especially nouns and named entities, between generated questions.
Use casual language.
Order the generated paraphrases in a numbered list."""
user = 'Generate 10 FAQ-style questions from the fact: "In the US, you are entitled to 30$ per day, minus any free meals which you choose to decline."'


prompt = f"""
### System:
{system}

### User:
{user}

### Assistant:"""

set_seed(42)
input_ids = tokenizer(prompt, return_tensors='pt').input_ids.cuda()
output = model.generate(inputs=input_ids, temperature=0.7, max_new_tokens=512)
print(tokenizer.decode(output[0]))


/mount/arbeitsdaten/asr-2/vaethdk/virtualenvs/cts_al/lib64/python3.10/site-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


<s> 
### System:
You are a helpful assistant creating a list of FAQ-style questions from given facts.
Only generate questions that can be answered by the given facts, without any external knowledge.
Remove some information, especially nouns and named entities, between generated questions.
Use casual language.
Order the generated paraphrases in a numbered list.

### User:
Generate 10 FAQ-style questions from the fact: "In the US, you are entitled to 30$ per day, minus any free meals which you choose to decline."

### Assistant:
1. What is the daily allowance for meals in the US?
2. Are there any deductions from the daily meal allowance?
3. What happens if you decline free meals?
4. Is the daily meal allowance the same for everyone in the US?
5. Are there any exceptions to the daily meal allowance rule?
6. Can you receive more than 30$ per day for meals in the US?
7. Are there any additional benefits or discounts for meals in the US?
8. Can you choose to decline free meals without affect

## Generate Question Synonyms

In [8]:
def generate_prompt(system: str, user: str) -> str:
    return f"""
    ### System:
    {system}

    ### User:
    {user}

    ### Assistant:"""

def generate_output(prompt: str, temperature: float = 0.7, max_new_tokens: int = 512) -> torch.FloatTensor:
    input_ids = tokenizer(prompt, return_tensors='pt').input_ids.cuda()
    output = model.generate(inputs=input_ids, temperature=temperature, max_new_tokens=max_new_tokens)
    return tokenizer.decode(output[0])


In [18]:
human_data_train = ReimburseGraphDataset('en/reimburse/train_graph.json', 'en/reimburse/train_answers.json', False, augmentation=DataAugmentationLevel.NONE, resource_dir="../../../resources/")

LOADING GRAPH...
GRAPH LOADED
LOADING ANSWERS FROM ../../../resources/en/reimburse/train_answers.json...
- not using synonyms
ANSWERS LOADED
===== Dataset Statistics =====
- files:  en/reimburse/train_graph.json en/reimburse/train_answers.json
- synonyms: False
- depth: 20  - degree: 13
- answers: 81
- questions: 279
- loaded original data: True
- loaded generated data: False
- question limit: 0  - maximum loaded:  7
- answer limit: 0  - maximum loaded:  1


In [19]:
# check that we don't have any answer synonyms
for node in human_data_train.node_list:
    for question in node.questions:
        human_data_train.question_list.remove(question)
        del human_data_train.questions_by_key[question.key]
    node.questions.clear()
assert len(human_data_train.question_list) == 0
assert len(human_data_train.questions_by_key) == 0

In [20]:
def parse_output(original_question: str, prompt: str, output: str, num_paraphrases: int) -> List[str]:
    # remove prompt from output first (ends at ### ASSISTANT: )
    questions = []
    cleaned = output[len(prompt):]
    
    if not "1." in cleaned: 
        print("NO LIST FOR QUESTION", original_question)
        return questions
    
    for i in range(1, num_paraphrases+1):
        if not f"{i}." in cleaned: 
            print(f" - NO {i}. CANDIDATE FOR QUESTION", original_question)
            continue

        start_idx = cleaned.find(f"{i}.") # find i. line
        end_idx = cleaned.find("\n", start_idx) # read until line end 
        if i == num_paraphrases and end_idx == -1:
            # last line might not have line break
            end_idx = len(cleaned)
        if start_idx == -1 or end_idx == -1:
            print(f" - INDEX PROBLEM FOR {i}. CANDIDATE: ({start_idx}, {end_idx})")
            continue
        # parse answer
        questions.append(cleaned[start_idx:end_idx].replace("</s>", "").strip())

        cleaned = cleaned[end_idx:] # remove i. line
    return questions

# V1

In [11]:
from data.dataset import NodeType, Question
import time

set_seed(42)

system = """You are a helpful assistant creating a list of FAQ-style questions from given facts.
Only generate questions that can be answered by the given facts, without any external knowledge.
Remove some information, especially nouns and named entities, between generated questions.
Use casual language.
Order the generated paraphrases in a numbered list."""

def user(answer_text: str, num_paraphrases: int) -> str:
    return f'Generate {num_paraphrases} FAQ-style questions from the fact: "{answer_text}"'

NUM_QUESTIONS = 10
TEMPERATURE = 0.7
MAX_NEW_TOKENS = 1024
generated_data = {}

for node in tqdm(human_data_train.nodes_by_type[NodeType.INFO]):
    prompt = generate_prompt(system=system, user=user(node.text, NUM_QUESTIONS))
    gen = generate_output(prompt=prompt, temperature=TEMPERATURE, max_new_tokens=MAX_NEW_TOKENS)
    candidates = parse_output(original_question=node.text, prompt=prompt, output=gen, num_paraphrases=NUM_QUESTIONS)
    for candidate in candidates:
        key = str(time.time()).replace(".", "")
        generated_data[key] = {
            "dialog_node_key": node.key,
            "key": key,
            "text": candidate,
        }

  0%|          | 0/80 [00:00<?, ?it/s]

100%|██████████| 80/80 [9:40:49<00:00, 435.62s/it]  


In [12]:
import json

cleaned_data = {}
for key in generated_data:
    node = human_data_train.nodes_by_key[generated_data[key]['dialog_node_key']]
    cleaned_data[key] = generated_data[key]
    for i in range (1, NUM_QUESTIONS+1):
        cleaned_data[key]['text'] = cleaned_data[key]['text'].replace(f"{i}.", "").strip()
    cleaned_data[key]["node_text"] = node.text
    cleaned_data[key]["node_type"] = node.node_type.value

with open("resources/en/reimburse/generated/train_questions_v1.json", "w") as f:
    json.dump(cleaned_data, f)

# DATA GENERATION V2: SHORTER QUESTIONS


In [36]:
from data.dataset import NodeType, Question
import time

set_seed(42)

system = """You are a helpful assistant creating a list of diverse FAQ-style questions from given facts.
Only generate questions that can be answered by the given facts, without any external knowledge.
Use casual language.
Prefer short questions.
Order the generated paraphrases in a numbered list."""

def user(answer_text: str, num_paraphrases: int) -> str:
    return f'Generate {num_paraphrases} short and diverse FAQ-style questions from the fact: "{answer_text}"'

NUM_QUESTIONS = 200
TEMPERATURE = 0.7
MAX_NEW_TOKENS = 24000
generated_data = {}

for node in tqdm(human_data_train.nodes_by_type[NodeType.INFO]):
    prompt = generate_prompt(system=system, user=user(node.text, NUM_QUESTIONS))
    gen = generate_output(prompt=prompt, temperature=TEMPERATURE, max_new_tokens=MAX_NEW_TOKENS)
    candidates = parse_output(original_question=node.text, prompt=prompt, output=gen, num_paraphrases=NUM_QUESTIONS)
    for candidate in candidates:
        key = str(time.time()).replace(".", "")
        generated_data[key] = {
            "dialog_node_key": node.key,
            "key": key,
            "text": candidate,
        }

  0%|          | 0/80 [00:00<?, ?it/s]

This is a friendly reminder - the current text generation call will exceed the model's predefined maximum length (2048). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.


 - INDEX PROBLEM FOR 98. CANDIDATE: (5, -1)
 - NO 99. CANDIDATE FOR QUESTION Flights are reimbursable there is a compelling  business or economic reason, e.g.: Ability to attend multiple sequential meetings 
 Health reasons 
 Saving work time 
 Flying is less expensive 
 - NO 100. CANDIDATE FOR QUESTION Flights are reimbursable there is a compelling  business or economic reason, e.g.: Ability to attend multiple sequential meetings 
 Health reasons 
 Saving work time 
 Flying is less expensive 
 - NO 101. CANDIDATE FOR QUESTION Flights are reimbursable there is a compelling  business or economic reason, e.g.: Ability to attend multiple sequential meetings 
 Health reasons 
 Saving work time 
 Flying is less expensive 
 - NO 102. CANDIDATE FOR QUESTION Flights are reimbursable there is a compelling  business or economic reason, e.g.: Ability to attend multiple sequential meetings 
 Health reasons 
 Saving work time 
 Flying is less expensive 
 - NO 103. CANDIDATE FOR QUESTION Flights are

KeyboardInterrupt: 

In [14]:
import json

cleaned_data = {}
for key in generated_data:
    node = human_data_train.nodes_by_key[generated_data[key]['dialog_node_key']]
    cleaned_data[key] = generated_data[key]
    for i in range (1, NUM_QUESTIONS+1):
        cleaned_data[key]['text'] = cleaned_data[key]['text'].replace(f"{i}.", "").strip()
    cleaned_data[key]["node_text"] = node.text
    cleaned_data[key]["node_type"] = node.node_type.value

with open("resources/en/reimburse/generated/llama/thesis/train_questions_200.json", "w") as f:
    json.dump(cleaned_data, f)

# DATA GENERATION V3: SHORTER QUESTIONS + SPLIT NODE CONTEXT

1. Try to generate shorter questions (-> change prompt)
2. Try to generate more diverse questions
    1. Detect relevant sentences in node text via NER tool (also detects time, quantities, ...)
    2. Generate questions for whole node context, then for only relevant sentences / sub-sentences of node
    3. Choose amount of questions to be generated depending on amount of extracted NERs?

In [8]:
# !pip install stanza

     |████████████████████████████████| 802 kB 14.3 MB/s eta 0:00:01
     |████████████████████████████████| 361 kB 119.2 MB/s eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
    Preparing wheel metadata ... done
  Created wheel for emoji: filename=emoji-2.7.0-py2.py3-none-any.whl size=356563 sha256=d6fa52fd4eea49ae84298a10e2c77bf642c90e75abcdd4db626c8b0309a7e8f2
  Stored in directory: /home/users2/vaethdk/.cache/pip/wheels/41/11/48/5df0b9727d5669c9174a141134f10304d1d78a3b89a4676f3d
Successfully built emoji
You should consider upgrading via the '/fs/scratch/users/vaethdk/adviser_reisekosten/.env/bin/python -m pip install --upgrade pip' command.


In [21]:
import stanza

In [22]:
torch.cuda.device_count()

1

In [10]:
# stanza.download(lang="en", model_dir=".models/")

2023-08-04 16:21:45 INFO: Downloading default packages for language: en (English) ...
2023-08-04 16:21:56 INFO: Finished downloading models and saved to .models/.


In [23]:
nlp = stanza.Pipeline('en', processors='tokenize,ner', device="cuda:0")

2025-01-21 13:13:36 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2025-01-21 13:13:36 INFO: Downloaded file to /home/users2/vaethdk/stanza_resources/resources.json
2025-01-21 13:13:36 WARNING: Language en package default expects mwt, which has been added
2025-01-21 13:13:37 INFO: Loading these models for language: en (English):
| Processor | Package                   |
-----------------------------------------
| tokenize  | combined                  |
| mwt       | combined                  |
| ner       | ontonotes-ww-multi_charlm |

2025-01-21 13:13:37 INFO: Using device: cuda:0
2025-01-21 13:13:37 INFO: Loading: tokenize
/mount/arbeitsdaten/asr-2/vaethdk/virtualenvs/cts_al/lib64/python3.10/site-packages/stanza/models/tokenization/trainer.py:82: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.m

In [25]:
from statistics import mean

nodes_with_ner = 0
nodes_without_ner = 0
avg_node_ner = []

for node in tqdm(human_data_train.nodes_by_type[NodeType.INFO]):
    context = nlp(node.text)
    if len(context.ents) > 0:
        nodes_with_ner += 1
        avg_node_ner.append(len(context.ents))
    else:
        nodes_without_ner += 1

print("TOTAL INFO NODES", len(human_data_train.nodes_by_type[NodeType.INFO]))
print("NODES WITH NER", nodes_with_ner)
print("NODES WITHOUT NER", nodes_without_ner)
print("AVG NER PER NODE WITH NER", mean(avg_node_ner))

  0%|          | 0/80 [00:00<?, ?it/s]

TOTAL INFO NODES 80
NODES WITH NER 30
NODES WITHOUT NER 50
AVG NER PER NODE WITH NER 2.1


In [26]:
avg_node_sentence_length = []

for node in human_data_train.nodes_by_type[NodeType.INFO]:
    avg_node_sentence_length.append(node.text.count("."))

print("MAX #SENTENCES PER NODE", max(avg_node_sentence_length))
print("AVG #SENTENCES PER NODE", mean(avg_node_sentence_length))

MAX #SENTENCES PER NODE 6
AVG #SENTENCES PER NODE 1.8


In [27]:
def extract_ner_sentences(node: DialogNode) -> List[Tuple[str, str]]:
    """
    Extract all sentences from node text that mention NER's.
    Returns them as a list of tuples, where each tuple contains
        1. the name of the entity
        2. the sentence containing that entity
    """
    results = []
    context = nlp(node.text)
    entities = context.ents
    for entity in entities:
        start_idx = entity.start_char
        end_idx = entity.end_char
        # expand start index to beginning of sentence
        while start_idx > 0 and node.text[start_idx-1] != ".":
            start_idx -= 1
        # expand end index to end of sentence
        while end_idx < len(node.text) and node.text[end_idx-1] != ".":
            end_idx += 1
        results.append((entity.text, node.text[start_idx:end_idx]))
    return results

In [29]:
# find a testing candidate
for node in human_data_train.nodes_by_type[NodeType.INFO]:
    results = extract_ner_sentences(node)
    if len(results) > 1:
        print(results)
        break

[('COVID-19', 'Please check the current COVID-19 travel warnings travel restrictions from the foreign ministry and the RKI.'), ('RKI', 'Please check the current COVID-19 travel warnings travel restrictions from the foreign ministry and the RKI.'), ('Department 4 (Administrative Department', ' In In extreme cases, authorization can be given by the leadership of Department 4 (Administrative Department).')]


In [30]:
print(node.text)

Please check the current COVID-19 travel warnings travel restrictions from the foreign ministry and the RKI. Business trips to high risk areas or virus variation areas are not generally not allowed. In In extreme cases, authorization can be given by the leadership of Department 4 (Administrative Department).


In [ ]:
from data.dataset import NodeType, Question
import time

system = """You are a helpful assistant creating a list of diverse FAQ-style questions from given facts.
Only generate questions that can be answered by the given facts, without any external knowledge.
Use casual language.
Prefer short questions.
Order the generated paraphrases in a numbered list."""

def user(answer_text: str, num_paraphrases: int) -> str:
    return f'Generate {num_paraphrases} short and diverse FAQ-style questions from the fact: "{answer_text}"'



NUM_QUESTIONS = 3
TEMPERATURE = 0.7
MAX_NEW_TOKENS = 1024
generated_data = {}


for entity, sentence in tqdm(extract_ner_sentences(node)):
    prompt = generate_prompt(system=system, user=user(sentence, NUM_QUESTIONS))
    gen = generate_output(prompt=prompt, temperature=TEMPERATURE, max_new_tokens=MAX_NEW_TOKENS)
    candidates = parse_output(original_question=node.text, prompt=prompt, output=gen, num_paraphrases=NUM_QUESTIONS)
    generated_data[entity] = candidates


  0%|          | 0/3 [00:00<?, ?it/s]

In [35]:
import pprint
pprint.pprint(generated_data)

{'COVID-19': ['1. What should I do to stay updated on travel warnings and '
              'restrictions related to COVID-19?',
              '2. Where can I find the latest information on travel advisories '
              'for COVID-19?',
              '3. Which organizations should I refer to for updates on '
              'COVID-19 travel restrictions?'],
 'Department 4 (Administrative Department': ['1. Who can give authorization in '
                                             'extreme cases?',
                                             "2. Which department's leadership "
                                             'can grant permission in extreme '
                                             'situations?',
                                             "3. What department's leadership "
                                             'has the power to authorize '
                                             'things in extreme cases?'],
 'RKI': ['1. What should I do to stay updated 

In [33]:
from data.dataset import NodeType, Question
import time

system = """You are a helpful assistant creating a list of diverse FAQ-style questions from given facts.
Only generate questions that can be answered by the given facts, without any external knowledge.
Use casual language.
Prefer short questions.
Order the generated paraphrases in a numbered list."""

def user(answer_text: str, num_paraphrases: int) -> str:
    return f'Generate {num_paraphrases} short and diverse FAQ-style questions from the fact: "{answer_text}"'


NUM_QUESTIONS = 3
TEMPERATURE = 0.7
MAX_NEW_TOKENS = 5000

prompt = generate_prompt(system=system, user=user(node.text, NUM_QUESTIONS))
gen = generate_output(prompt=prompt, temperature=TEMPERATURE, max_new_tokens=MAX_NEW_TOKENS)
candidates = parse_output(original_question=node.text, prompt=prompt, output=gen, num_paraphrases=NUM_QUESTIONS)
pprint.pprint(candidates)

['1. What should I do to check the current travel warnings and restrictions '
 'related to COVID-19?',
 '2. Are business trips to high risk or virus variation areas generally '
 'allowed?',
 '3. Can the leadership of Department 4 authorize trips to high risk areas in '
 'extreme cases?']


In [20]:
system = """You are a helpful assistant creating a list of diverse FAQ-style questions from given facts.
Only generate questions that can be answered by the given facts, without any external knowledge.
Use casual language.
Prefer short questions.
Order the generated paraphrases in a numbered list."""

def user(answer_text: str, num_paraphrases: int) -> str:
    return f'Generate {num_paraphrases} short and diverse FAQ-style questions from the fact: "{answer_text}"'

def user_ner(answer_text: str, ner: str, num_paraphrases: int) -> str:
    return f'Generate {num_paraphrases} short and diverse FAQ-style questions about the entity "{ner}" from the fact: "{answer_text}"'




NUM_QUESTIONS = 10
NUM_QUESTIONS_PER_SENTENCE = 3
TEMPERATURE = 0.7
MAX_NEW_TOKENS = 1024
generated_data = {}

set_seed(42)

for node in tqdm(human_data_train.nodes_by_type[NodeType.INFO]):
    # use dict indexed by generated text to filter out duplicates
    all_generations = {}
    
    # extract NERs
    named_entities = extract_ner_sentences(node)

    # Generate questions with NER sentences only, make asking about NER a requirement
    for entity, sentence in named_entities:
        prompt = generate_prompt(system=system, user=user_ner(node.text, entity, NUM_QUESTIONS_PER_SENTENCE))
        gen = generate_output(prompt=prompt, temperature=TEMPERATURE, max_new_tokens=MAX_NEW_TOKENS)
        candidates = parse_output(original_question=node.text, prompt=prompt, output=gen, num_paraphrases=NUM_QUESTIONS_PER_SENTENCE)
        for candidate_idx, candidate in enumerate(candidates):
            key = str(time.time()).replace(".", "")
            cleaned_candidate = candidate.replace(f"{candidate_idx+1}.", "").strip()
            all_generations[cleaned_candidate] = {
                "context": "ner",
                "entity": entity,
                "dialog_node_key": node.key,
                "key": key,
                "text": cleaned_candidate
            }

    # Generate questions with whole context
    num_node_level_questions = max(NUM_QUESTIONS_PER_SENTENCE, NUM_QUESTIONS - len(named_entities) * NUM_QUESTIONS_PER_SENTENCE)
    prompt = generate_prompt(system=system, user=user(node.text, num_node_level_questions))
    gen = generate_output(prompt=prompt, temperature=TEMPERATURE, max_new_tokens=MAX_NEW_TOKENS)
    candidates = parse_output(original_question=node.text, prompt=prompt, output=gen, num_paraphrases=NUM_QUESTIONS_PER_SENTENCE)
    for candidate_idx, candidate in enumerate(candidates):
        key = str(time.time()).replace(".", "")
        cleaned_candidate = candidate.replace(f"{candidate_idx+1}.", "").strip()
        all_generations[cleaned_candidate] = {
            "context": "node",
            "dialog_node_key": node.key,
            "key": key,
            "text": cleaned_candidate
        }
    
    # add filtered questions to generated dataset
    for text in all_generations:
        entry = all_generations[text]
        generated_data[entry["key"]] = entry

100%|██████████| 80/80 [12:37:21<00:00, 568.01s/it]  


In [21]:
import json

cleaned_data = {}
for key in generated_data:
    node = human_data_train.nodes_by_key[generated_data[key]['dialog_node_key']]
    cleaned_data[key] = generated_data[key]
    for i in range (1, NUM_QUESTIONS+1):
        cleaned_data[key]['text'] = cleaned_data[key]['text'].replace(f"{i}.", "").strip()
    cleaned_data[key]["node_text"] = node.text
    cleaned_data[key]["node_type"] = node.node_type.value

with open("resources/en/reimburse/generated/train_questions_v3.json", "w") as f:
    json.dump(cleaned_data, f)